# Phi-3.5-mini-instruct — automated hyperparameter search (Google Colab)

**Model:** [`microsoft/Phi-3.5-mini-instruct`](https://huggingface.co/microsoft/Phi-3.5-mini-instruct), approximately **3.8 billion parameters**.

### What it does

1. Uses the **existing saved 1,000-ticket teacher-labeled Bitext dataset** and the same cluster-level train/validation/test split (seed 42). **No new teacher labels are generated.**
2. Tests two Phi-3.5 configurations in **quick** mode (`2e-4` and `1e-4`), five in **extended** mode (additional lower LR, 2 epochs, and LoRA rank 32). Custom trials also supported.
3. Trains with **4-bit NF4 QLoRA**, LoRA on `qkv_proj`, `o_proj`, `gate_up_proj`, `down_proj`, a clean Phi baseline, and Phi-specific chat formatting from its tokenizer.
4. Runs the **existing LLM judge**, ROUGE-L and business/latency evaluations. Ranks by **validation judge composite**, never by test score. Optionally compares against the previously saved SmolLM2 validation summary **only when shared dataset and split signatures match**.
5. Stores separate, content-fingerprinted runs under `MyDrive/slm-distillation/experiments/hpo_phi35_*` and results under `experiments/hpo_phi35_search/`. Completed runs are skipped. Incomplete outputs are not overwritten.

### Run instructions

1. In Colab select **Runtime → Change runtime type → GPU** (T4 ~16 GB minimum; L4/A100 more comfortable). Run setup through compatibility patch and optional tokenizer preflight.
2. In **Search controls**, leave `DRY_RUN=True` for a no-training preview, run the dataset and plan cells, then the execution cell.
3. To actually train, set `DRY_RUN=False`, initially set `MAX_NEW_RUNS_PER_EXECUTION=1`, rerun the controls, plan, and execution cells. Each trial can use external **Anthropic API credits**; full inference on all 37 clusters is much slower than for SmolLM2.
4. Run **Compare validation results** and **Inspect selected winner**. The summary files are saved to Drive.

**Important:** This notebook has been statically validated, but **not executed on a Colab GPU or against live API keys**. The GitHub branch, its installed dependencies and external API can change. The guard patches deliberately stop if the expected source code is missing. Phi-3.5-mini is a 3.8B model, not a 17B model; larger model = potentially greater memory/runtime requirements. Do not interpret tiny validation differences from ~37 clusters as a decisive result.

**Sources:** [Microsoft model card](https://huggingface.co/microsoft/Phi-3.5-mini-instruct), [project trainer](https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation/blob/btt-setup-js-test/code/phase1/finetuning/trainer.py).
**v2 installation diagnostic:** The dependency cell now shows the actual pip resolver/build error and saves the full log to `/content/phi35_pip_install.log`. Python 3.13 compatibility is not assumed; do not change pinned versions blindly before reading the error.


## 1. Environment setup — run these cells once per Colab session

In [ ]:
# ── 1. GPU check ───────────────────────────────────────────────────────────────
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime (Runtime → Change runtime type → GPU).')
name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU: {name}; {vram_gb:.1f} GiB; bf16 support: {torch.cuda.is_bf16_supported()}')
if vram_gb < 14:
    raise RuntimeError('Phi-3.5-mini 4-bit QLoRA needs a larger GPU. Use T4 16GB, L4 or A100.')
print('Note: T4 may be tight during long sequences/evaluation. Start with batch size 1.')

Wed Nov 11 03:59:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ── CELL 2: Mount Google Drive ────────────────────────────────────────────────
# A popup asks for permissions — click Allow on everything (normal Google behaviour).
from google.colab import drive
import os
drive.mount('/content/drive')
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive mounted')
else:
    print('❌ Mount failed — run this cell again')

Mounted at /content/drive
✅ Drive mounted


In [ ]:
# ── Clone or update your project's checkout (without putting the PAT in Git URL) ──
# The existing repo was confirmed to be public at the time this notebook was built.
# If your branch becomes private, authenticate with GitHub separately before cloning.
from pathlib import Path
import os, subprocess, sys

BRANCH_NAME = 'btt-setup-js-test'  # Must match the branch used by your original notebook.
REPO_ORG = 'Break-Through-Tech'
REPO_NAME = 'Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation'
REPO_DIR = Path('/content/project')
REPO_URL = f'https://github.com/{REPO_ORG}/{REPO_NAME}.git'

def git_run(args, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f'Git command failed: {args[:3]}\n{result.stderr[-2000:]}')
    return result.stdout.strip()

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        raise RuntimeError('Existing /content/project is not a Git checkout. Inspect it before removing it.')
    git_run(['git', 'clone', '--branch', BRANCH_NAME, REPO_URL, str(REPO_DIR)])
else:
    # Repair remote URL if an earlier notebook stored credentials in origin URL.
    git_run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_DIR)
    git_run(['git', 'fetch', 'origin', BRANCH_NAME], cwd=REPO_DIR)
    git_run(['git', 'checkout', BRANCH_NAME], cwd=REPO_DIR)
    git_run(['git', 'pull', '--ff-only', 'origin', BRANCH_NAME], cwd=REPO_DIR)

if (REPO_DIR / 'code' / 'main.py').exists():
    CODE_DIR = REPO_DIR / 'code'
elif (REPO_DIR / 'main.py').exists():
    CODE_DIR = REPO_DIR
else:
    raise FileNotFoundError(f'main.py not found in {REPO_DIR} or code/')

os.environ['SLM_CODE_DIR'] = str(CODE_DIR)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print('Branch:', BRANCH_NAME)
print('Git revision:', git_run(['git','rev-parse','HEAD'], cwd=REPO_DIR))
print('Code directory:', CODE_DIR)

Branch: btt-setup-js-test
Git revision: 09629936c58a17b92807ad422ef1c755132c3196
Code directory: /content/project/code


In [ ]:
# ── 4. Install project dependencies (diagnostic output; Python 3.13 aware) ──
# The first notebook hid pip's useful error lines with -q and only showed a
# CalledProcessError. This version captures the complete pip output to a file,
# and prints the most relevant tail on failure. It does not silently change
# major training-library versions or skip required dependencies.
# TRL 0.12.2 requires transformers <4.47.0, so 4.46.3 is pinned.
# The previous pin 4.48.3 was incompatible and caused ResolutionImpossible.
from pathlib import Path
import subprocess, sys, platform, importlib.metadata as md

code = Path(os.environ['SLM_CODE_DIR'])
req = code / 'requirements.txt'
assert req.exists(), f'Missing {req}'
constraints = Path('/content/phi35_constraints.txt')
constraints.write_text('''transformers==4.46.3
trl==0.12.2
peft==0.14.0
accelerate==1.3.0
datasets>=2.21.0,<4.0.0
''')

log_path = Path('/content/phi35_pip_install.log')
cmd = [sys.executable, '-m', 'pip', 'install', '--prefer-binary',
       '--progress-bar', 'off', '-r', str(req), '-c', str(constraints),
       'bitsandbytes>=0.45.0,<0.50.0']
print('Python:', sys.version.split()[0], 'on', platform.platform())
print('Requirements:', req)
print('Constraints (fixes trl 0.12.2 / transformers 4.48.3 conflict):\n' + constraints.read_text())
print('Installing; full output will be written to', log_path)
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, errors='replace')
log_path.write_text(result.stdout or '', encoding='utf-8')
if result.returncode != 0:
    output_lines = (result.stdout or '').splitlines()
    print('\nPIP FAILED with exit code', result.returncode)
    print('---- Last 100 output lines ----')
    print('\n'.join(output_lines[-100:]) or '(No pip output received)')
    print('---- End of pip output ----')
    print('The failure above identifies the real package/resolver issue.')
    print('If it is not clear, share the last 100 lines from:', log_path)
    raise RuntimeError('Dependency installation failed. See pip output ABOVE (not just traceback).')

print('\nInstallation succeeded. Installed versions:')
for pkg in ['transformers','trl','peft','accelerate','bitsandbytes','datasets']:
    print(' ', pkg, md.version(pkg))
print('If Colab asks to restart the runtime, do so, then rerun setup in order.')


Python: 3.13.16 on Linux-6.6.159+-x86_64-with-glibc2.39
Requirements: /content/project/code/requirements.txt
Constraints (fixes trl 0.12.2 / transformers 4.48.3 conflict):
transformers==4.46.3
trl==0.12.2
peft==0.14.0
accelerate==1.3.0
datasets>=2.21.0,<4.0.0

Installing; full output will be written to /content/phi35_pip_install.log

Installation succeeded. Installed versions:
  transformers 4.46.3
  trl 0.12.2
  peft 0.14.0
  accelerate 1.3.0
  bitsandbytes 0.49.2
  datasets 3.6.0
If Colab asks to restart the runtime, do so, then rerun setup in order.


In [ ]:
# ── 5. Drive layout / persistent Phi cache ─────────────────────────────────────
import os
DRIVE_ROOT = '/content/drive/MyDrive/slm-distillation'
HF_CACHE_PHI = f'{DRIVE_ROOT}/hf_cache_phi35'
for folder in [f'{DRIVE_ROOT}/data/raw', f'{DRIVE_ROOT}/data/processed',
               f'{DRIVE_ROOT}/data/checkpoints', f'{DRIVE_ROOT}/outputs',
               f'{DRIVE_ROOT}/shared_datasets', f'{DRIVE_ROOT}/experiments',
               HF_CACHE_PHI]:
    os.makedirs(folder, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_PHI
os.environ['DRIVE_ROOT'] = DRIVE_ROOT
print('Drive root:', DRIVE_ROOT)
print('Persistent Phi-3.5 cache:', HF_CACHE_PHI)
print('The initial official model download may require roughly 8 GB of Drive space.')

Drive root: /content/drive/MyDrive/slm-distillation
Persistent Phi-3.5 cache: /content/drive/MyDrive/slm-distillation/hf_cache_phi35
The initial official model download may require roughly 8 GB of Drive space.


In [ ]:
# ── 6. Colab Secrets; no keys embedded in the notebook ────────────────────────
from google.colab import userdata
import os

def load_secret(name, required=False):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
        print(f'  ✅ {name} available')
        return True
    print(f'  {"❌" if required else "ℹ️ "} {name}: not set')
    return not required

print('Anthropic judge requires ANTHROPIC_API_KEY when DRY_RUN=False.')
print('HF_TOKEN is optional for this public Microsoft model.')
load_secret('ANTHROPIC_API_KEY', required=False)
load_secret('HF_TOKEN', required=False)
load_secret('OPENAI_API_KEY', required=False)

Anthropic judge requires ANTHROPIC_API_KEY when DRY_RUN=False.
HF_TOKEN is optional for this public Microsoft model.
  ✅ ANTHROPIC_API_KEY available
  ✅ HF_TOKEN available
  ℹ️  OPENAI_API_KEY: not set


True

In [ ]:
# ── 7. Verify runtime + local prerequisites ───────────────────────────────────
from pathlib import Path
import torch, os
code = Path(os.environ.get('SLM_CODE_DIR', '/content/project/code'))
checks = {
  'GPU >=14 GiB': torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory / (2**30) >= 14,
  'Google Drive mounted': Path('/content/drive/MyDrive').exists(),
  'Repository main.py': (code / 'main.py').is_file(),
  'Trainer source': (code / 'phase1/finetuning/trainer.py').is_file(),
  'Pipeline source': (code / 'phase1/pipeline.py').is_file(),
  'Base config': (code / 'configs/phase1_config.yaml').is_file(),
  'Persistent HF cache set': os.environ.get('HF_HOME', '').endswith('/hf_cache_phi35'),
}
for name, ok in checks.items():
    print(('✅' if ok else '❌'), name)
assert all(checks.values()), 'Fix prerequisites before proceeding.'
print('ANTHROPIC_API_KEY:', 'configured' if os.environ.get('ANTHROPIC_API_KEY') else 'missing (required for real evaluation)')

✅ GPU >=14 GiB
✅ Google Drive mounted
✅ Repository main.py
✅ Trainer source
✅ Pipeline source
✅ Base config
✅ Persistent HF cache set
ANTHROPIC_API_KEY: configured


---
### Phi-3.5-specific compatibility patch (run once per Colab session)

The upstream repository already resolves the correct **fused Phi projections**, but its generic Unsloth loader attaches a LoRA adapter during model loading. For valid baseline results this notebook forces **standard Transformers + PEFT** for Phi-3.5, defers adapter attachment to training, selects fp16 quantized compute on T4, and uses eager attention for compatibility. It also avoids loading the 3.8B model merely to build a tokenized dataset.

The patch **changes only the local Colab checkout**, never your GitHub remote. It uses exact-match guards and stops if the code differs from the expected branch. Inspect the output before launching trials. The source version is incorporated into every trial signature.

In [ ]:
# ── 8. Guarded, reversible local source patch ─────────────────────────────────
from pathlib import Path
import ast, os
CODE = Path(os.environ.get('SLM_CODE_DIR', '/content/project/code'))
trainer = CODE / 'phase1/finetuning/trainer.py'
pipeline = CODE / 'phase1/pipeline.py'
assert trainer.is_file() and pipeline.is_file(), 'Clone the expected repository branch first.'

for p in (trainer, pipeline):
    backup = p.with_suffix('.py.bak_phi35_setup')
    if not backup.exists():
        backup.write_text(p.read_text())

def guarded_replace(path, before, after, label):
    s = path.read_text()
    if after in s:
        print('Already applied:', label)
        return
    if s.count(before) != 1:
        raise RuntimeError(f'{label}: expected exactly one matching section in {path}; found {s.count(before)}. Stop and inspect.')
    new = s.replace(before, after, 1)
    ast.parse(new, filename=str(path))
    path.write_text(new)
    print('Applied:', label)

# Keep the loader from attaching an adapter for a supposedly clean baseline.
guarded_replace(trainer,
    '        model = FastLanguageModel.get_peft_model(\n            model,\n            r=lora_cfg["r"],\n            lora_alpha=lora_cfg["lora_alpha"],\n            target_modules=target_modules,\n            lora_dropout=lora_cfg["lora_dropout"],\n            bias=lora_cfg["bias"],\n            use_gradient_checkpointing="unsloth",\n        )\n'.replace('\\n','\n'),
    '        # Phi experiment: a baseline model must not already have a LoRA adapter.\n'.replace('\\n','\n'),
    'Never attach LoRA during base-model loading')
# Avoid Unsloth's alternative model repackaging and layer naming on Phi.
guarded_replace(trainer,
    '    try:\n        from unsloth import FastLanguageModel'.replace('\\n','\n'),
    '    try:\n        if "phi-3.5-mini" in model_id.lower():\n            raise ImportError("Use native Phi-3.5 model for reproducible QLoRA")\n        from unsloth import FastLanguageModel'.replace('\\n','\n'),
    'Use standard Transformers 4-bit loader for Phi')
# Detect actual PEFT wrapper and prepare quantized model for LoRA training.
guarded_replace(trainer,
    '    from peft import get_peft_model, LoraConfig, TaskType',
    '    from peft import get_peft_model, LoraConfig, TaskType, PeftModel, prepare_model_for_kbit_training',
    'Import adapter-aware training helpers')
guarded_replace(trainer,
    '    if not _is_unsloth_model(model):\n        peft_config = LoraConfig('.replace('\\n','\n'),
    '    if not isinstance(model, PeftModel):\n        if device_mode == "colab" and getattr(model, "is_loaded_in_4bit", False):\n            model = prepare_model_for_kbit_training(\n                model, use_gradient_checkpointing=train_cfg["gradient_checkpointing"]\n            )\n        peft_config = LoraConfig('.replace('\\n','\n'),
    'Prepare 4-bit weights before attaching one LoRA adapter')
guarded_replace(trainer,
    '        bnb_4bit_compute_dtype=torch.bfloat16,',
    '        bnb_4bit_compute_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16),',
    'Use fp16 compute on T4 / bf16 on capable GPUs')
guarded_replace(trainer,
    '        model_id, quantization_config=bnb_config, device_map="auto"',
    '        model_id, quantization_config=bnb_config, device_map="auto",\n        attn_implementation="eager", trust_remote_code=True'.replace('\\n','\n'),
    'Use compatible Phi eager attention on CUDA')
guarded_replace(trainer,
    '    tokenizer = AutoTokenizer.from_pretrained(model_id)\n    return model, tokenizer'.replace('\\n','\n'),
    '    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)\n    return model, tokenizer'.replace('\\n','\n'),
    'Trust official Microsoft tokenizer implementation')
guarded_replace(pipeline,
    '        _, tokenizer_tmp = load_model_and_tokenizer(cfg)',
    '        from transformers import AutoTokenizer\n        tokenizer_tmp = AutoTokenizer.from_pretrained(cfg["student_slm"]["model_id"].strip(), trust_remote_code=True)'.replace('\\n','\n'),
    'Do not load 3.8B model for dataset formatting')
guarded_replace(pipeline,
    "datetime.now().strftime('%Y%m%d_%H%M')",
    "datetime.now().strftime('%Y%m%d_%H%M%S')",
    'Use seconds in output run IDs')

for p in (trainer, pipeline):
    ast.parse(p.read_text(), filename=str(p))
assert 'qkv_proj", "o_proj", "gate_up_proj", "down_proj' in trainer.read_text(), 'Phi LoRA target resolver missing; stop.'
print('✅ Patches applied and both Python files pass syntax checks.')

Already applied: Never attach LoRA during base-model loading
Already applied: Use standard Transformers 4-bit loader for Phi
Already applied: Import adapter-aware training helpers
Already applied: Prepare 4-bit weights before attaching one LoRA adapter
Already applied: Use fp16 compute on T4 / bf16 on capable GPUs
Already applied: Use compatible Phi eager attention on CUDA


RuntimeError: Trust official Microsoft tokenizer implementation: expected exactly one matching section in /content/project/code/phase1/finetuning/trainer.py; found 3. Stop and inspect.

### Optional model/tokenizer preflight (small Hugging Face downloads only)

Set `RUN_TOKENIZER_PREFLIGHT=True` to verify that Phi-3.5's **model configuration, native chat template and tokenizer** load before starting a long experiment. This does **not** download the 7.6 GB model weights or invoke a paid judge. Leave it `False` if you only want to inspect the dry-run plan offline.

In [ ]:
# ── Optional config/tokenizer-only smoke test ─────────────────────────────────
RUN_TOKENIZER_PREFLIGHT = False   # change to True once; downloads config/tokenizer files
if RUN_TOKENIZER_PREFLIGHT:
    from transformers import AutoConfig, AutoTokenizer
    model_id = 'microsoft/Phi-3.5-mini-instruct'
    cfg_check = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    tok_check = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    assert 'phi' in cfg_check.model_type.lower(), f'Unexpected model type: {cfg_check.model_type}'
    text = tok_check.apply_chat_template([
        {'role':'system','content':'Give a short label.'},
        {'role':'user','content':'Cluster: password reset and login failures'}
    ], tokenize=False, add_generation_prompt=True)
    assert text and ('assistant' in text.lower()), 'Chat template did not include assistant prompt.'
    assert isinstance(tok_check.encode(text), list) and len(tok_check.encode(text)) > 0
    print('✅ Model type:',cfg_check.model_type)
    print('✅ Vocabulary:',len(tok_check),' | prompt tokens:',len(tok_check.encode(text)))
    print('✅ Chat-template preview:',repr(text[:180]))
else:
    print('Preflight skipped; set RUN_TOKENIZER_PREFLIGHT=True to verify the tokenizer/config.')

## 2. Search controls

**Quick** tests `2e-4` and `1e-4` (both 3 epochs, LoRA r=16/alpha=16): directly comparable to your existing SmolLM2 parameter settings. **Extended** adds `5e-5`, 2 epochs, and LoRA r=32. **Custom** uses the user-defined list below.

Start with **dry run** and **one real trial at a time**. Phi-3.5 needs substantially more VRAM/time than SmolLM2. A T4 is a reasonable minimum for 4-bit batch-1 training, **not a guarantee of an OOM-free run**; if needed lower `MAX_SEQ_LENGTH` to 1024, but doing so changes your experiment signature and affects comparability.

In [ ]:
# ── 9. Edit ordinary experiment controls here ────────────────────────────────
SEARCH_PRESET = 'quick'             # quick: 2 trials; extended: 5; custom: edit below
DRY_RUN = True                     # set to False only when ready to train and pay judge API calls
MAX_NEW_RUNS_PER_EXECUTION = 1     # safe default: one trial at a time
INCOMPLETE_ACTION = 'skip'         # 'skip' or 'archive_and_restart' (no checkpoint resume)
STOP_ON_ERROR = True

N_SAMPLES = 1000                   # fixed teacher-labeled dataset used by the Smol notebook
SPLIT_SEED = 42
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 16                    # effective microbatch accumulation = 16

MODEL_IDS = {'phi35': 'microsoft/Phi-3.5-mini-instruct'}
CUSTOM_TRIALS = [
    {'model':'phi35', 'label':'custom_lr_75e6', 'learning_rate':7.5e-5, 'epochs':3, 'lora_r':16, 'lora_alpha':16},
    {'model':'phi35', 'label':'custom_lr_1e4_rank32', 'learning_rate':1e-4, 'epochs':3, 'lora_r':32, 'lora_alpha':32},
]

assert SEARCH_PRESET in {'quick','extended','custom'}
assert INCOMPLETE_ACTION in {'skip','archive_and_restart'}
assert isinstance(DRY_RUN, bool)
assert isinstance(MAX_NEW_RUNS_PER_EXECUTION, int) and MAX_NEW_RUNS_PER_EXECUTION >= 0
assert MAX_SEQ_LENGTH >= 512 and BATCH_SIZE >= 1 and GRAD_ACCUM >= 1
print('Search:', SEARCH_PRESET, '| Dry run:', DRY_RUN, '| Max new runs:', MAX_NEW_RUNS_PER_EXECUTION)

## 3. Reuse the **same** teacher-labeled dataset as SmolLM2

No fresh labels and no reclustering. The notebook looks for `shared_datasets/bitext_n1000_teacher/data/processed` and verifies that it contains the three original input CSV files and **37 clusters**. Only if the shared dataset folder is missing does it consider a single, unambiguous saved 37-cluster source. Set `DATASET_SOURCE` explicitly if multiple candidates appear. No changing or overwriting existing shared data automatically.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, hashlib, json, yaml, pandas as pd
from datetime import datetime, timezone

CODE = Path(os.environ.get('SLM_CODE_DIR', '/content/project/code'))
REPO = Path('/content/project')
DRIVE_ROOT = Path('/content/drive/MyDrive/slm-distillation')
EXPERIMENTS = DRIVE_ROOT / 'experiments'
SEARCH_ROOT = EXPERIMENTS / 'hpo_phi35_search'
SHARED_DATA = DRIVE_ROOT / 'shared_datasets/bitext_n1000_teacher/data/processed'
REQUIRED_FILES = ('bitext_clustered.csv', 'bitext_grouped.csv', 'bitext_labeled.csv')
# Optional explicit source folder containing the REQUIRED_FILES, if shared data missing.
DATASET_SOURCE = None

assert (CODE / 'main.py').is_file(), 'Run environment setup and clone first.'
assert (CODE / 'configs/phase1_config.yaml').is_file(), 'Missing base config in this Git checkout.'
assert (DRIVE_ROOT / 'data').exists(), 'Mount Drive / run setup first.'

def all_required(folder):
    return all((folder / name).is_file() for name in REQUIRED_FILES)

if not all_required(SHARED_DATA):
    if DATASET_SOURCE:
        candidate = Path(DATASET_SOURCE)
        assert all_required(candidate), f'Explicit source lacks required CSVs: {candidate}'
    else:
        candidates = []
        for labeled in EXPERIMENTS.glob('*/data/processed/bitext_labeled.csv'):
            folder = labeled.parent
            if not all_required(folder):
                continue
            try:
                groups = pd.read_csv(folder / 'bitext_grouped.csv')
                if len(groups) == 37: # original 1,000-ticket snapshot
                    candidates.append(folder)
            except Exception:
                continue
        if len(candidates) != 1:
            raise RuntimeError(
                f'Shared dataset missing at {SHARED_DATA}. Found {len(candidates)} '
                'plausible candidates. Set DATASET_SOURCE to your trusted folder.\n' +
                '\n'.join(str(c) for c in candidates[:20]))
        candidate = candidates[0]
    SHARED_DATA.mkdir(parents=True, exist_ok=True)
    for p in sorted(candidate.iterdir()):
        if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.jsonl'}:
            shutil.copy2(p, SHARED_DATA / p.name)
    print('Created shared snapshot from:', candidate)

assert all_required(SHARED_DATA), f'Missing input file in {SHARED_DATA}'

def sha_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

dataset_hashes = {p.name: sha_file(p) for p in sorted(SHARED_DATA.iterdir())
                  if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.jsonl'}}
assert set(REQUIRED_FILES).issubset(dataset_hashes)
assert N_SAMPLES == 1000, 'This shared dataset is the original n1000 snapshot; use a separate snapshot for other sample counts.'
DATASET_SHA = hashlib.sha256(json.dumps(dataset_hashes,sort_keys=True).encode()).hexdigest()
BASE_YAML = CODE / 'configs/phase1_config.yaml'
BASE_CFG_SHA = sha_file(BASE_YAML)
BASE_CFG = yaml.safe_load(BASE_YAML.read_text())
GIT_SHA = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True,
                         capture_output=True, check=True).stdout.strip()
PATCHED_FILE_HASHES = {
    name: sha_file(CODE / name)
    for name in ['phase1/finetuning/trainer.py','phase1/pipeline.py']
}
print('Shared dataset:', SHARED_DATA)
cluster_count = len(pd.read_csv(SHARED_DATA / 'bitext_grouped.csv'))
assert cluster_count == 37, f'Expected 37 clusters in saved n1000 dataset, got {cluster_count}'
print('Cluster count:', cluster_count)
print('Dataset SHA256:', DATASET_SHA[:16])
print('Base config SHA256:', BASE_CFG_SHA[:16])
print('Git revision:', GIT_SHA[:12])

## 4. Build a deterministic Phi-3.5 search plan

Identical dataset, split, LoRA dropout and evaluation settings for all planned trials. Trial fingerprints include the dataset snapshot, Git revision, local compatibility patches, base configuration and hyperparameters. Results never share a directory with SmolLM2 runs.

In [ ]:
import copy, re, json, hashlib, pandas as pd

VARS = [
    {'label': 'lr_2e4',   'learning_rate':2e-4, 'epochs':3, 'lora_r':16, 'lora_alpha':16},
    {'label': 'lr_1e4',   'learning_rate':1e-4, 'epochs':3, 'lora_r':16, 'lora_alpha':16},
    {'label': 'lr_5e5',   'learning_rate':5e-5, 'epochs':3, 'lora_r':16, 'lora_alpha':16},
    {'label': 'epochs_2', 'learning_rate':1e-4, 'epochs':2, 'lora_r':16, 'lora_alpha':16},
    {'label': 'rank_32',  'learning_rate':1e-4, 'epochs':3, 'lora_r':32, 'lora_alpha':32},
]

if SEARCH_PRESET == 'quick':
    RAW_TRIALS = [{'model':model, **variant} for model in MODEL_IDS for variant in VARS[:2]]
elif SEARCH_PRESET == 'extended':
    RAW_TRIALS = [{'model':model, **variant} for model in MODEL_IDS for variant in VARS]
else:
    RAW_TRIALS = list(CUSTOM_TRIALS)
assert RAW_TRIALS, 'No trials configured for this search.'

KNOWN_TRIAL_KEYS = {'model','label','learning_rate','epochs','lora_r','lora_alpha','lora_dropout'}

def validate_trial(t):
    unknown = set(t) - KNOWN_TRIAL_KEYS
    assert not unknown, f'Unknown trial keys {unknown}: {t}'
    assert t['model'] in MODEL_IDS, f'Unknown model {t["model"]}'
    assert t['learning_rate'] > 0 and t['learning_rate'] <= 0.001
    assert isinstance(t['epochs'], int) and t['epochs'] >= 1
    assert isinstance(t['lora_r'], int) and t['lora_r'] in (4,8,16,32,64)
    assert isinstance(t['lora_alpha'], int) and t['lora_alpha'] > 0
    assert 0 <= t.get('lora_dropout',0) < 1


def canonical(obj): return json.dumps(obj, sort_keys=True, separators=(',', ':'), allow_nan=False)

TRIAL_PLAN = []
seen = set()
for trial in RAW_TRIALS:
    validate_trial(trial)
    t = dict(trial)
    t.setdefault('lora_dropout', BASE_CFG['lora'].get('lora_dropout', 0))
    # Disallow collisions where labels differ but actual params are identical.
    signature = {
        'model_id': MODEL_IDS[t['model']],
        'params': {k:t[k] for k in ('learning_rate','epochs','lora_r','lora_alpha','lora_dropout')},
        'shared': {'n_samples':N_SAMPLES,'seed':SPLIT_SEED,'max_seq_length':MAX_SEQ_LENGTH,
                   'batch_size':BATCH_SIZE,'grad_accum':GRAD_ACCUM},
        'source': {'dataset_sha':DATASET_SHA, 'base_cfg_sha':BASE_CFG_SHA,
                   'git_sha':GIT_SHA,'patches':PATCHED_FILE_HASHES},
    }
    digest = hashlib.sha256(canonical(signature).encode()).hexdigest()[:12]
    if digest in seen:
        raise ValueError('Duplicate model/hyperparameter combination: ' + repr(t))
    seen.add(digest)
    safe_label = re.sub('[^a-z0-9_]+','_',t['label'].lower())[:32].strip('_')
    exp_name = f'hpo_{t["model"]}_{safe_label}_{digest}'
    TRIAL_PLAN.append({'trial':t,'signature':signature,'experiment':exp_name,
                       'folder':EXPERIMENTS / exp_name})

plan_table = pd.DataFrame([
    {'experiment':r['experiment'],'model':r['trial']['model'],
     'lr':r['trial']['learning_rate'], 'epochs':r['trial']['epochs'],
     'LoRA r':r['trial']['lora_r'], 'LoRA alpha':r['trial']['lora_alpha'],
     'existing outputs':(r['folder'] / 'outputs').exists()}
    for r in TRIAL_PLAN
])
print(f'Planned trials: {len(TRIAL_PLAN)} | new trials allowed this execution: {MAX_NEW_RUNS_PER_EXECUTION}')
display(plan_table)

## 5. Execute sequentially (or preview without any training)

The existing `main.py --phase 1` runs the **same dataset builder, clean-base inference, LoRA training, fine-tuned inference, validation judge, ROUGE and business evaluation**. Real runs require `ANTHROPIC_API_KEY`, run one at a time, and log into separate Drive folders.

`DRY_RUN=True` does **not** stage data, download model weights, train, or call paid APIs. The setup itself may install packages. Restarting Colab and repeating the setup/controls/dataset/plan steps skips complete runs; incomplete outputs are **preserved**, not silently resumed. `archive_and_restart` explicitly archives only the old trial output folder before starting from scratch.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import copy, json, os, shutil, subprocess, sys, time, yaml, pandas as pd, torch

EXPECTED_CSVS = ('judge_summary.csv','metrics_summary.csv','business_eval.csv')
VAL_ALIASES = {'val','valid','validation','dev'}

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def complete_eval_dirs(outputs):
    if not outputs.is_dir(): return []
    found=[]
    for run in sorted(outputs.iterdir()):
        ev=run/'evaluation'
        if run.is_dir() and all((ev/name).is_file() and (ev/name).stat().st_size > 0
                                    for name in EXPECTED_CSVS):
            found.append(ev)
    return found

def eval_has_validation(ev):
    try:
        j = pd.read_csv(ev / 'judge_summary.csv')
        ok = ('split' in j and 'model' in j and 'composite' in j and
              ((j['split'].astype(str).str.lower().str.strip().isin(VAL_ALIASES)) &
               (j['model'].astype(str).str.lower().str.strip() == 'finetuned')).any())
        return bool(ok)
    except Exception:
        return False

def outputs_state(dest):
    outs = dest/'outputs'
    if not outs.exists(): return 'NEW', None
    if not outs.is_dir(): return 'INCOMPLETE', None
    valid = complete_eval_dirs(outs)
    if valid:
        # 'Complete' here = comparison CSVs exist; validation may still be missing.
        newest = max(valid, key=lambda ev: ev.stat().st_mtime)
        return ('COMPLETE' if eval_has_validation(newest) else 'COMPLETE_NO_VALIDATION'), newest
    return 'INCOMPLETE', None

def build_config(rec):
    trial=rec['trial']
    cfg=copy.deepcopy(BASE_CFG)
    for sect in ('training','lora','qlora','student_slm','dataset','colab','paths','pipeline','evaluation'):
        assert isinstance(cfg.get(sect), dict), f'Base config missing section: {sect}'
    cfg['device_mode']='colab'
    cfg['student_slm']['model_id']=MODEL_IDS[trial['model']]
    cfg['student_slm']['max_seq_length']=MAX_SEQ_LENGTH
    cfg['dataset']['n_samples']=N_SAMPLES
    cfg['seed']=SPLIT_SEED
    cfg['training'].update({
        'num_train_epochs':trial['epochs'],
        'learning_rate':float(trial['learning_rate']),
        'per_device_train_batch_size':BATCH_SIZE,
        'gradient_accumulation_steps':GRAD_ACCUM,
        'fp16':not torch.cuda.is_bf16_supported(),'bf16':torch.cuda.is_bf16_supported(),
    })
    cfg['lora'].update({'r':trial['lora_r'], 'lora_alpha':trial['lora_alpha'],
                        'lora_dropout':trial['lora_dropout']})
    cfg['qlora']['load_in_4bit']=True
    cfg['colab']['drive_root_override']=str(rec['folder'])
    cfg['colab']['cache_model_on_drive']=False
    cfg['paths']['hf_cache']=str(DRIVE_ROOT / 'hf_cache_phi35')
    cfg['pipeline'].update({
        'run_clustering':False, 'run_preprocessing':False, 'run_label_generation':False,
        'run_finetuning':True, 'run_baseline_eval':True, 'run_finetuned_eval':True,
        'run_llm_judge':True, 'run_business_eval':True,
    })
    cfg['evaluation'].update({
        'existing_run_dir':None, 'eval_all_splits':True, 'run_bertscore':False,
    })
    return cfg

def stage_trial(rec):
    dest=rec['folder']
    dest.mkdir(parents=True,exist_ok=True)
    payload=yaml.safe_dump(build_config(rec),sort_keys=False)
    saved=dest/'experiment_config.yaml'
    local_cfg=CODE/'configs'/f'{rec["experiment"]}.yaml'
    manifest_file=dest/'manifest.json'
    expected_manifest={'name':rec['experiment'], 'trial':rec['trial'],
                       'signature':rec['signature'], 'config_sha256':hashlib.sha256(payload.encode()).hexdigest(),
                       'source_dataset':str(SHARED_DATA)}
    if saved.exists() and saved.read_text()!=payload:
        raise RuntimeError(f'Config drift detected at {saved}. Never overwrite an existing trial.')
    if manifest_file.exists() and json.loads(manifest_file.read_text()) != expected_manifest:
        raise RuntimeError(f'Manifest mismatch at {manifest_file}. Inspect before continuing.')
    if local_cfg.exists() and local_cfg.read_text()!=payload:
        raise RuntimeError(f'Local config collision at {local_cfg}.')
    if not saved.exists(): saved.write_text(payload)
    if not manifest_file.exists(): manifest_file.write_text(json.dumps(expected_manifest,indent=2))
    local_cfg.write_text(payload)
    processed=dest/'data/processed'
    processed.mkdir(parents=True,exist_ok=True)
    for src in sorted(SHARED_DATA.iterdir()):
        if not src.is_file() or src.suffix.lower() not in {'.csv','.json','.jsonl'}:
            continue
        target=processed/src.name
        if target.exists():
            if target.stat().st_size!=src.stat().st_size or sha_file(target)!=sha_file(src):
                raise RuntimeError(f'Copied dataset differs from shared snapshot: {target}')
        else:
            shutil.copy2(src,target)
    return local_cfg

def write_status(rec, status, **extra):
    info={'experiment':rec['experiment'], 'status':status, 'updated_utc':now_utc(), **extra}
    (rec['folder']/'run_status.json').write_text(json.dumps(info,indent=2,default=str))

STARTED=0
for index,rec in enumerate(TRIAL_PLAN,1):
    state, ev = outputs_state(rec['folder'])
    print(f'[{index}/{len(TRIAL_PLAN)}] {rec["experiment"]} -> {state}')
    if state in ('COMPLETE','COMPLETE_NO_VALIDATION'):
        if state == 'COMPLETE_NO_VALIDATION':
            print('  WARNING: CSVs found but no validation judge row. Comparison cannot select this trial.')
        continue
    if state == 'INCOMPLETE' and INCOMPLETE_ACTION == 'skip':
        print('  SKIPPED: incomplete outputs preserved. Inspect logs/checkpoints.')
        continue
    if DRY_RUN:
        print('  DRY RUN: would stage and train this trial; no writes or model/API calls.')
        continue
    if STARTED >= MAX_NEW_RUNS_PER_EXECUTION:
        print('  LIMIT reached; run this cell again to advance the queue.')
        break
    if not os.environ.get('ANTHROPIC_API_KEY'):
        raise RuntimeError('ANTHROPIC_API_KEY missing: real trials run paid Anthropic judge calls. Configure Colab Secrets.')
    cfg_path=stage_trial(rec)
    if state == 'INCOMPLETE':
        old_outputs=rec['folder']/'outputs'
        archive=rec['folder']/('outputs_archive_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
        assert old_outputs.exists() and not archive.exists()
        old_outputs.rename(archive)
        print('  Archived incomplete outputs:',archive)
    write_status(rec,'RUNNING',command=f'{sys.executable} {CODE / "main.py"} --phase 1 --config {cfg_path} --device_mode colab')
    STARTED += 1
    cmd=[sys.executable,str(CODE/'main.py'),'--phase','1','--config',str(cfg_path), '--device_mode','colab']
    log_file=rec['folder']/'pipeline.log'
    print('  Starting training + evaluation. Log:',log_file)
    try:
        t0=time.monotonic()
        with open(log_file,'a',encoding='utf-8') as out:
            out.write('\n=== '+now_utc()+' ===\n')
            out.flush()
            proc=subprocess.Popen(cmd, cwd=str(CODE),env=os.environ.copy(),
                                  stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                  text=True,errors='replace',bufsize=1)
            assert proc.stdout is not None
            for line in proc.stdout:
                print(line,end='')
                out.write(line)
                out.flush()
            returncode=proc.wait()
        minutes=(time.monotonic()-t0)/60
    except BaseException as exc:
        write_status(rec,'INTERRUPTED',error=repr(exc))
        raise
    if returncode != 0:
        write_status(rec,'FAILED',returncode=returncode, wall_minutes=minutes)
        print('  FAILED:',returncode,'| saved outputs remain untouched')
        if STOP_ON_ERROR:
            raise RuntimeError(f'Failed: {rec["experiment"]}; inspect {log_file}')
        continue
    final_state, final_ev=outputs_state(rec['folder'])
    write_status(rec,final_state,returncode=returncode,wall_minutes=minutes,
                 evaluation_dir=str(final_ev) if final_ev else None)
    print('  Exit 0,',final_state, '|',round(minutes,1),'min')
    if final_state == 'INCOMPLETE':
        print('  WARNING: pipeline returned success but expected evaluation CSVs are missing.')
        if STOP_ON_ERROR:
            raise RuntimeError('Missing expected evaluation files; inspect pipeline.log')

print('New trials launched in this execution:',STARTED)

## 6. Rank only using validation judge results

A trial can enter the leaderboard only if all three expected evaluation CSVs are available and `judge_summary.csv` contains a **fine-tuned validation row**. The winner uses the same tie-breaking logic as your SmolLM2 notebook: judge composite, then faithfulness, then lower latency. Results are exported to `experiments/hpo_phi35_search/`.

**Interpretation:** 37 cluster groups means the validation subset is very small. A marginal improvement may be sampling or judge noise. Run fresh seeds/held-out clusters before making strong conclusions.

In [ ]:
from pathlib import Path
import pandas as pd, json

rows=[]
for rec in TRIAL_PLAN:
    state, ev = outputs_state(rec['folder'])
    t=rec['trial']
    row={
        'experiment':rec['experiment'],'model':t['model'],
        'learning_rate':t['learning_rate'],'epochs':t['epochs'],
        'lora_r':t['lora_r'],'lora_alpha':t['lora_alpha'],
        'state':state,'judge_val':float('nan'), 'faithfulness_val':float('nan'),
        'specificity_val':float('nan'),'rouge_l_val':float('nan'),
        'latency_sec':float('nan'), 'train_minutes':float('nan'),
        'evaluation_dir':str(ev) if ev else ''
    }
    if ev and state == 'COMPLETE':
        try:
            judge=pd.read_csv(ev/'judge_summary.csv')
            judge['split_clean']=judge['split'].astype(str).str.lower().str.strip()
            judge['model_clean']=judge['model'].astype(str).str.lower().str.strip()
            j=judge[judge['split_clean'].isin(VAL_ALIASES) & (judge['model_clean']=='finetuned')]
            assert len(j)==1, f'Expected 1 validation finetuned judge row, got {len(j)}'
            for dest,src in [('judge_val','composite'),('faithfulness_val','faithfulness'),
                             ('specificity_val','specificity')]:
                if src in j.columns: row[dest]=float(j.iloc[0][src])
            metrics=pd.read_csv(ev/'metrics_summary.csv')
            if {'split','model','rouge_l_same'}.issubset(metrics.columns):
                m=metrics[metrics['split'].astype(str).str.lower().str.strip().isin(VAL_ALIASES) &
                          (metrics['model'].astype(str).str.lower().str.strip()=='finetuned')]
                if len(m)==1: row['rouge_l_val']=float(m.iloc[0]['rouge_l_same'])
            biz=pd.read_csv(ev/'business_eval.csv')
            if {'metric','value'}.issubset(biz.columns):
                b=dict(zip(biz['metric'],pd.to_numeric(biz['value'],errors='coerce')))
                row['latency_sec']=b.get('slm_inference_latency_mean_s_finetuned',float('nan'))
                row['train_minutes']=b.get('finetuning_wall_time_min',float('nan'))
        except Exception as exc:
            row['state']='EVALUATION_PARSE_ERROR'
            row['error']=str(exc)
    rows.append(row)

summary=pd.DataFrame(rows)
valid=summary[(summary['state']=='COMPLETE') & pd.notna(summary['judge_val'])].copy()
valid=valid.sort_values(['judge_val','faithfulness_val','latency_sec'],
                        ascending=[False,False,True],na_position='last')

print('All planned experiments:')
display(summary.drop(columns=['evaluation_dir']).round(4))
print('\nRankable experiments (validation judge):')
if valid.empty:
    print('None yet. If CSVs exist but are COMPLETE_NO_VALIDATION, your pipeline must emit validation scores.')
else:
    display(valid[['experiment','model','learning_rate','epochs','lora_r','lora_alpha',
                   'judge_val','faithfulness_val','specificity_val','rouge_l_val',
                   'latency_sec','train_minutes']].round(4))

SEARCH_ROOT.mkdir(parents=True,exist_ok=True)
summary.to_csv(SEARCH_ROOT/'search_summary.csv',index=False)
print('Saved:',SEARCH_ROOT/'search_summary.csv')
if not valid.empty:
    top=valid.iloc[0]
    winner={
        'selection':'highest observed validation judge composite; tie-break faithfulness then latency',
        'experiment':top['experiment'], 'model':top['model'],
        'judge_val':float(top['judge_val']),
        'config':str(EXPERIMENTS/top['experiment']/'experiment_config.yaml'),
        'evaluation_dir':top['evaluation_dir'],
        'ranked_count':len(valid),
        'selected_utc':now_utc(),
        'caveat':'validation-only selection, not a definitive held-out test result',
    }
    (SEARCH_ROOT/'best_experiment.json').write_text(json.dumps(winner,indent=2))
    print('Best observed validation trial:',top['experiment'],'score:',top['judge_val'])
    print('Saved:',SEARCH_ROOT/'best_experiment.json')
else:
    old=SEARCH_ROOT/'best_experiment.json'
    if old.exists():
        print('WARNING: previous best_experiment.json exists; it may be stale for this plan.')

## 7. Inspect the chosen configuration and its held-out test result

Only **after choosing the best validation trial** does this cell show its test-split metrics. Do not use test scores to tune further trial hyperparameters, or the test split stops being independent.

In [ ]:
if valid.empty:
    print('No validation-ranked winner; no test results shown.')
else:
    best=valid.iloc[0]
    ev=Path(best['evaluation_dir'])
    print('Selected from validation only:',best['experiment'])
    print('Config:',EXPERIMENTS/best['experiment']/'experiment_config.yaml')
    for filename in ['judge_summary.csv','metrics_summary.csv']:
        df=pd.read_csv(ev/filename)
        if 'split' not in df:
            print(filename,': no split column')
            continue
        test=df[df['split'].astype(str).str.lower().str.strip().eq('test')]
        print('\n',filename,'(test only):')
        if test.empty:
            print('No test rows saved.')
        else:
            display(test)

## 8. Optional: compare the saved Phi results with your earlier SmolLM2 results

This is a **read-only** comparison with `hpo_smol_search/search_summary.csv`, if it is present. Comparison is shown **only when each candidate’s run manifest says the dataset SHA, cluster split fractions/seed, top-k, teacher, max sequence length, batch and gradient accumulation match**. Otherwise it reports the mismatch instead of presenting misleading rankings. Differences in model architecture, repository patches and wall-clock conditions are *not* fully controlled.

In [ ]:
# ── Optional cross-model validation comparison; no new GPU/API work ───────────
from pathlib import Path
import json, pandas as pd, yaml
smol_csv = EXPERIMENTS / 'hpo_smol_search' / 'search_summary.csv'
phi_csv = SEARCH_ROOT / 'search_summary.csv'

if not smol_csv.exists() or not phi_csv.exists():
    print('Comparison unavailable: run Phi summary cell first and ensure Smol summary exists.')
else:
    def compatible_summary(csv_path, source):
        df = pd.read_csv(csv_path)
        out=[]
        for _, r in df.iterrows():
            if r.get('state') != 'COMPLETE' or pd.isna(r.get('judge_val')):
                continue
            exp_dir = EXPERIMENTS / str(r['experiment'])
            manifest = exp_dir / 'manifest.json'
            config_path = exp_dir / 'experiment_config.yaml'
            if not manifest.exists() or not config_path.exists():
                print('Not comparable (no manifest/config):', r['experiment']); continue
            try:
                data = json.loads(manifest.read_text())
                sig = data['signature']
                saved_cfg = yaml.safe_load(config_path.read_text())
                if (saved_cfg.get('split') != BASE_CFG.get('split') or
                    saved_cfg.get('top_k') != BASE_CFG.get('top_k') or
                    saved_cfg.get('teacher_llm',{}).get('model') != BASE_CFG.get('teacher_llm',{}).get('model')):
                    print('Not comparable (split/top_k/teacher mismatch):', r['experiment'])
                    continue
                p = sig['shared']
                expected = {'n_samples':N_SAMPLES, 'seed':SPLIT_SEED,
                            'max_seq_length':MAX_SEQ_LENGTH,
                            'batch_size':BATCH_SIZE, 'grad_accum':GRAD_ACCUM}
                if sig['source']['dataset_sha'] != DATASET_SHA or any(p.get(k)!=v for k,v in expected.items()):
                    print('Not comparable (dataset/split/config mismatch):',r['experiment'])
                    continue
                row=r[['experiment','model','learning_rate','epochs','lora_r',
                       'judge_val','faithfulness_val','specificity_val',
                       'rouge_l_val','latency_sec','train_minutes']].to_dict()
                row['source']=source
                out.append(row)
            except Exception as exc:
                print('Cannot verify:',r['experiment'],repr(exc))
        return out

    both = compatible_summary(smol_csv,'SmolLM2') + compatible_summary(phi_csv,'Phi-3.5')
    if both:
        leaderboard=pd.DataFrame(both).sort_values('judge_val',ascending=False)
        display(leaderboard[['source','model','experiment','learning_rate','epochs','lora_r',
                             'judge_val','faithfulness_val','specificity_val',
                             'rouge_l_val','latency_sec','train_minutes']].round(4))
        print('CAUTION: Matching data and training settings do not eliminate differences in code or GPU timing.')
    else:
        print('No results with verified matching dataset and shared settings.')

## Practical notes / constraints

- **Official checkpoint:** `microsoft/Phi-3.5-mini-instruct`, 3.8B parameters, native **Phi-3.5 fused projections** `qkv_proj`, `o_proj`, `gate_up_proj`, `down_proj`. The repo already knows these targets; the patch keeps a **clean baseline** and a single adapter for the fine-tuned model.
- **GPU/memory:** 4-bit NF4, eager attention, batch 1 and 2048-token cap. A **T4 16 GB is a minimum target, not a guarantee**; L4/A100 may work better. If OOM, use `MAX_SEQ_LENGTH=1024`, acknowledging this changes the benchmark. Do not blindly increase batch size.
- **Software:** installation pins Transformers 4.46.3 / TRL 0.12.2, because TRL 0.12.2 requires `transformers<4.47.0`. Python 3.13 and recent CUDA/PyTorch remain unverified on live Colab; inspect import/preflight results before training. Only the public official model remote code is trusted. Recheck these versions if Colab changes.
- **Security:** Keys are read from Colab Secrets and never stored in the notebook. Downloading model code with `trust_remote_code=True` executes code from Microsoft's model repo; review it if your environment is sensitive. No GitHub credentials are embedded.
- **Data integrity:** Relies on an existing, verified 1,000-ticket/37-cluster teacher-labeled snapshot. Dataset and source hashes ensure runs are identified deterministically. The same seed does not guarantee identical optimization across model architectures.
- **Spend:** Training runs external Anthropic judge requests, even though teacher label generation is disabled. Use `DRY_RUN=True` first; expect substantially slower runs than your SmolLM2 baseline. No time or cost estimate is guaranteed.
- **Run bookkeeping:** Results under `experiments/hpo_phi35_search/search_summary.csv` and `best_experiment.json`. Finished runs are skipped. `archive_and_restart` is a fresh train, **not** checkpoint resume.
- **Validation vs test:** All three splits may be evaluated internally; only the validation split is ranked. The held-out test split must not influence hyperparameter choice. For trustworthy comparisons, also inspect actual labels and repeat with other splits/seeds.
- **Comparison to SmolLM2:** Optional side-by-side summary cell checks matching dataset SHA, seed, and train/val/test settings before showing a leaderboard. It will not silently compare mismatched inputs.

**Not live-tested:** This file was checked for valid `.ipynb` structure and Python syntax locally, but GPU training, the Hugging Face download, the GitHub checkout and the paid judge cannot be executed inside the notebook-authoring environment.

9. Export all saved per-prediction evaluations to your 17-column CSV
Run this cell after mounting Google Drive. It does not need a GPU, and it does not retrain or re-judge anything. It scans saved Phi-3.5 (and optionally SmolLM2) HPO trial folders and writes:

MyDrive/slm-distillation/experiments/model_evaluation_detail.csv

It joins prediction JSONL, original evaluation/_cache judge/non-LLM/latency CSVs, the saved ticket snapshot, and each experiment's config. One CSV row = one model variant × one support-ticket cluster × one prompt P1–P5. Batch Size = number of tickets included in that prompt (top-k, usually 10), not the training batch size. Batch identifies the cluster and the available train/val/test split.
- Actual Prompt contains the reconstructed system/user message content. Tokenizer-generated special chat tokens are not included because they were not saved with historical predictions.
- Cosine Score, ROUGE, and BertScore use the multi-reference version when available.
- An individual judge score may be missing when the judge sampled only a subset of predictions.
- Relevance is empty because your current rubric measures equivalence/coherence, not relevance. BertScore was disabled in this notebook. Per-prediction Estimated/Approx Cost is unavailable and remains empty.
- The model's output is a cluster theme label, not a solution to an individual support ticket.
Only data from an experiment's own output folder are joined; an unobserved result stays blank. If no predictions exist yet, the CSV will have headers only. Requires PyYAML, which the notebook's installation uses. To download the generated file in Colab, use the Files pane or Google Drive.

In [ ]:
"""Export saved Phase 1 cluster-label evaluations into the requested 17-column CSV.

Read-only for all experiment files; writes only the requested output CSV.
No GPU, model inference, retraining or judge calls.
"""
from __future__ import annotations

import csv
import json
from collections import Counter
from pathlib import Path

import yaml

COLUMNS = [
    'Name of Model', 'Model Owner', 'Prompt ID', 'Actual Prompt',
    'Input to the Model', 'Output of the Model', 'Latency', 'Batch',
    'Batch Size', 'Faithfulness', 'Relevance', 'Specificity', 'Composite',
    'Cosine Score', 'ROUGE', 'BertScore', 'Estimated/Approx Cost'
]


def read_csv(path):
    if not path.is_file(): return []
    with path.open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))


def read_jsonl(path):
    if not path.is_file(): return []
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def first(*values):
    for x in values:
        if x is not None and str(x).strip().lower() not in ('', 'none', 'null', 'nan'):
            return x
    return ''


def as_text(obj):
    if isinstance(obj, str): return obj
    if obj is None: return ''
    return json.dumps(obj, ensure_ascii=False)


def records_by_key(rows, path):
    result = {}
    for record in rows:
        cid = first(record.get('cluster_id'))
        pid = first(record.get('prompt_id'))
        if cid == '' or pid == '':
            raise ValueError(f'Cannot join {path}: missing cluster_id or prompt_id')
        key = (str(cid).strip(), str(pid).strip())
        if key in result:
            raise ValueError(f'Cannot join {path}: duplicate record {key}')
        result[key] = record
    return result


def unpack_json(record, column):
    value = record.get(column)
    if not value: return record
    if isinstance(value, dict): return {**record, **value}
    try:
        extracted = json.loads(value)
    except (ValueError, TypeError):
        return record
    return {**record, **extracted} if isinstance(extracted, dict) else record


def discover_evaluations(root, include_smol=True):
    """Include root/outputs and experiments/*/outputs, irrespective of folder names/config."""
    found = []
    paths = [root.parent / 'outputs']
    paths.extend(sorted(p / 'outputs' for p in root.iterdir() if p.is_dir()))
    for outputs in paths:
        if not outputs.is_dir(): continue
        for run in sorted(outputs.iterdir()):
            ev = run / 'evaluation'
            if not ev.is_dir(): continue
            if not any((ev / (tag + '_predictions.jsonl')).is_file()
                       for tag in ('baseline', 'finetuned')):
                continue
            if not include_smol:
                descriptor = (str(run.name) + ' ' + str(outputs.parent.name)).lower()
                if 'phi' not in descriptor: continue
            found.append((outputs.parent, run, ev))
    return found


def load_run_config(parent, run, ev):
    candidates = [
        parent / 'experiment_config.yaml',
        parent / 'experiment_config.yml',
        parent / 'config.yaml',
        run / 'experiment_config.yaml',
        run / 'config.yaml',
        run / 'config_used.yaml',
        run / 'pipeline_config.yaml',
        ev / 'config.yaml',
    ]
    for path in candidates:
        if path.is_file():
            cfg = yaml.safe_load(path.read_text(encoding='utf-8'))
            if isinstance(cfg, dict): return cfg, path
    return {}, None


def infer_model_id(cfg, run):
    saved_id = first((cfg.get('student_slm') or {}).get('model_id'))
    if saved_id: return str(saved_id)
    name = run.name.lower()
    if 'phi-3.5-mini' in name or 'phi35' in name:
        return 'microsoft/Phi-3.5-mini-instruct'
    if 'smollm2-1.7b' in name or 'smol17b' in name or '_17b' in name:
        return 'HuggingFaceTB/SmolLM2-1.7B-Instruct'
    if 'smollm2-360m' in name or 'smol360' in name or '_360m' in name:
        return 'HuggingFaceTB/SmolLM2-360M-Instruct'
    return ''


def owner_of(model_id):
    return {'microsoft': 'Microsoft', 'huggingfacetb': 'HuggingFaceTB'}.get(
        model_id.split('/')[0].lower(), model_id.split('/')[0] if '/' in model_id else '')


def read_tickets(parent, run, root, cfg):
    """Use a run-specific snapshot; do not substitute another experiment's dataset."""
    candidates = [
        run / 'labeled_data.csv',
        run / 'data/processed/bitext_labeled.csv',
        parent / 'data/processed/bitext_labeled.csv',
    ]
    # For root-level runs only, the original root-level processed dataset is eligible.
    if parent == root.parent:
        candidates.append(root.parent / 'data/processed/bitext_labeled.csv')
    source = next((p for p in candidates if p.is_file()), None)
    if source is None: return {}, None

    # The number of tickets actually included in inference prompts must be known.
    k = first(cfg.get('top_k'), (cfg.get('dataset') or {}).get('top_k'))
    try: k = int(k)
    except (ValueError, TypeError): return {}, source
    if k <= 0: return {}, source

    collected = {}
    for row in read_csv(source):
        try:
            cid = str(row['cluster_id']).strip()
            rank = int(float(row['ticket_rank_in_cluster']))
            details = str(row['ticket_details']).strip()
        except (KeyError, TypeError, ValueError):
            continue
        if rank <= k and details:
            collected.setdefault(cid, []).append((rank, details))
    return {cid: [text for _, text in sorted(rows)] for cid, rows in collected.items()}, source


def reconstruct_prompt(cfg, pid, tickets):
    prompts = cfg.get('prompts') or {}
    if not tickets or not isinstance(prompts, dict) or not prompts.get('system') or pid not in prompts:
        return ''
    system = str(prompts['system']).strip()
    if prompts.get('include_domain_in_prompt') and (cfg.get('dataset') or {}).get('domain'):
        system = 'You are analysing ' + str(cfg['dataset']['domain']) + ' support tickets. ' + system
    ticket_block = '\n'.join(f'Ticket {i+1}: {s.strip()}' for i, s in enumerate(tickets))
    user_text = str(prompts[pid]).replace('{tickets}', ticket_block)
    return '[SYSTEM]\n' + system + '\n[USER]\n' + user_text


def judge_metrics(record):
    if str(record.get('reasoning', '')).strip().lower() == 'parse_error':
        return '', '', ''
    values = [first(record.get('faithfulness')),
              first(record.get('specificity')),
              first(record.get('composite_score'), record.get('composite'))]
    return tuple('' if str(v) in ('0', '0.0') else v for v in values)


def split_lookup(ev):
    mapping = {}
    for row in read_csv(ev / 'labels_by_cluster.csv'):
        cid, split = first(row.get('cluster_id')), first(row.get('split'))
        if cid == '' or split == '': continue
        cid, split = str(cid).strip(), str(split).strip().lower()
        if cid in mapping and mapping[cid] != split:
            raise ValueError(f'Conflicting splits for cluster {cid}: {ev}')
        mapping[cid] = split
    return mapping


def cache_lookup(ev, tag, choices, blob_column):
    for filename in choices:
        path = ev / '_cache' / filename.format(tag=tag)
        if path.is_file():
            records = records_by_key(read_csv(path), path)
            return {key: unpack_json(row, blob_column) for key, row in records.items()}, path
    return {}, None


def export(root, out, include_smol=True):
    root = Path(root)
    out = Path(out)
    if not root.is_dir(): raise FileNotFoundError(root)
    sources = discover_evaluations(root, include_smol=include_smol)
    rows = []
    stats = {'evaluation_folders': len(sources), 'prediction_files': 0,
             'runs_without_config': [], 'runs_without_judge_cache': [],
             'runs_without_metrics_cache': [], 'runs_without_ticket_reconstruction': [],
             'by_variant': Counter(), 'missing_fields': Counter()}

    for parent, run, ev in sources:
        cfg, cfg_source = load_run_config(parent, run, ev)
        if not cfg_source: stats['runs_without_config'].append(str(ev))
        model_id = infer_model_id(cfg, run)
        ticket_map, ticket_source = read_tickets(parent, run, root, cfg)
        if not ticket_map: stats['runs_without_ticket_reconstruction'].append(str(ev))
        splits = split_lookup(ev)
        run_rel = str(run.relative_to(root.parent))

        for tag in ('baseline', 'finetuned'):
            pred_file = ev / f'{tag}_predictions.jsonl'
            preds = read_jsonl(pred_file)
            if not preds: continue
            stats['prediction_files'] += 1
            by_key = records_by_key(preds, pred_file)
            judge, judge_path = cache_lookup(
                ev, tag, ('judge_scores_{tag}.csv', 'llm_{tag}.csv'), 'llm_metrics')
            metrics, metric_path = cache_lookup(
                ev, tag, ('{tag}_metrics.csv', 'nonllm_{tag}.csv'), 'nonllm_metrics')
            latency, _ = cache_lookup(
                ev, tag, ('latency_{tag}.csv',), '_unused_blob_column_')
            if judge_path is None: stats['runs_without_judge_cache'].append(f'{run_rel}/{tag}')
            if metric_path is None: stats['runs_without_metrics_cache'].append(f'{run_rel}/{tag}')
            for (cid, pid), p in by_key.items():
                tick = ticket_map.get(cid, [])
                input_block = '\n'.join(f'Ticket {i+1}: {text}' for i, text in enumerate(tick))
                j = judge.get((cid, pid), {})
                m = metrics.get((cid, pid), {})
                l = latency.get((cid, pid), {})
                faith, spec, comp = judge_metrics(j)
                name = f'{model_id} ({tag}; {run_rel})' if model_id else f'{tag}; {run_rel}'
                row = {
                    'Name of Model': name,
                    'Model Owner': owner_of(model_id),
                    'Prompt ID': pid,
                    'Actual Prompt': first(p.get('actual_prompt'), p.get('full_prompt'),
                                           reconstruct_prompt(cfg, pid, tick)),
                    'Input to the Model': first(p.get('input_to_model'), p.get('input_text'), input_block),
                    'Output of the Model': first(p.get('generated_label'), p.get('output')),
                    'Latency': first(l.get('latency_s'), p.get('latency_s')),
                    'Batch': f'{run_rel} | cluster_{cid}' + (f' [{splits[cid]}]' if cid in splits else ''),
                    # Ticket count in the cluster prompt, not training microbatch size.
                    'Batch Size': len(tick) if tick else '',
                    'Faithfulness': faith,
                    'Relevance': '',   # Existing judge measures other dimensions.
                    'Specificity': spec,
                    'Composite': comp,
                    'Cosine Score': first(m.get('cosine_sim_multi_ref'), m.get('cosine_sim_multi')),
                    'ROUGE': first(m.get('rougeL_multi_ref'), m.get('rouge_l_multi')),
                    'BertScore': first(m.get('bertscore_f1_multi_ref'), m.get('bertscore_f1_multi')),
                    'Estimated/Approx Cost': '',  # Cannot assign run-level cost to rows.
                }
                rows.append(row)
                stats['by_variant'][tag] += 1
                for col, val in row.items():
                    if val == '': stats['missing_fields'][col] += 1

    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    stats['rows'] = len(rows)
    stats['output'] = str(out)
    return stats




# Run after the existing Google Drive mounting cell.
EXP = Path('/content/drive/MyDrive/slm-distillation/experiments')
CSV_OUT = EXP / 'model_evaluation_detail.csv'
report = export(EXP, CSV_OUT, include_smol=True)
print('CSV saved to:', report['output'])
print('Evaluation folders found:', report['evaluation_folders'])
print('Prediction files read:', report['prediction_files'])
print('Rows exported:', report['rows'])
print('By variant:', dict(report['by_variant']))
print('Missing fields:', dict(report['missing_fields']))
for key in ('runs_without_config', 'runs_without_judge_cache',
            'runs_without_metrics_cache', 'runs_without_ticket_reconstruction'):
    print(f'{key}: {len(report[key])}')
    for example in report[key][:3]: print(' ', example)
if report['rows'] == 0:
    print('No matching predictions were found, so CSV contains only headers.')


CSV saved to: /content/drive/MyDrive/slm-distillation/experiments/model_evaluation_detail.csv
Evaluation folders found: 14
Prediction files read: 28
Rows exported: 2200
By variant: {'baseline': 1100, 'finetuned': 1100}
Missing fields: {'Actual Prompt': 650, 'Input to the Model': 650, 'Batch Size': 650, 'Relevance': 2200, 'BertScore': 2200, 'Estimated/Approx Cost': 2200, 'Faithfulness': 1190, 'Specificity': 1190, 'Composite': 1190}
runs_without_config: 9
  /content/drive/MyDrive/slm-distillation/outputs/20260929_1629_SmolLM2-360M-Instruct_ep3/evaluation
  /content/drive/MyDrive/slm-distillation/outputs/20260929_1656_SmolLM2-360M-Instruct_ep5/evaluation
  /content/drive/MyDrive/slm-distillation/outputs/20260929_2027_Phi-3.5-mini-instruct_ep3/evaluation
runs_without_judge_cache: 0
runs_without_metrics_cache: 0
runs_without_ticket_reconstruction: 9
  /content/drive/MyDrive/slm-distillation/outputs/20260929_1629_SmolLM2-360M-Instruct_ep3/evaluation
  /content/drive/MyDrive/slm-distillation/